## tl;dr

HP는 기본 8시간, 최대 10시간. 추천 HP 지수는 0.8, 기준은 6,000이다. 독립 48개 경로에서 12시간 접속 간격의 직업 성장속도 격차는 5.5062%로, 이전 12시간 상한안 10.6188%보다 작았다. 운영 적용은 하지 않았다.


## Context & Methods

### Key Assumptions

MP·DEX·CHA는 기존 상한 곡선을 유지한다. HP만 96개 곡선을 보정 경로 seed 2–5에서 비교하고, seed 6–13은 별도로 검증한다. 상대 성장속도는 같은 목표 레벨까지 평균 소요시간의 역수다. 접속 주기는 엔진 전투 경로에 시간저장량을 환산한 근사이며 실제 플레이어 로그가 아니다.

Jupyter 패키지가 없어 코드 셀은 일반 Python으로 순서대로 실행했다. Jupyter 커널 자체의 실행은 미검증이다. 재실행하려면 pandas/numpy 및 nbconvert/ipykernel이 있는 환경에서 `python -m jupyter nbconvert --execute --to notebook --inplace ten_hour_review.ipynb`를 실행한다.


## Data

### 1. Load verified local trajectories


In [1]:
from pathlib import Path
import sys,json,itertools
import numpy as np
import pandas as pd
analysis_root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'tools/analysis/ten_hour_balance.py').exists())
sys.path.insert(0,str(analysis_root/'tools/analysis'))
import ten_hour_balance as balance
balance.OUT=analysis_root/'docs/audits/2026-08-31-stat-bonus-balance/cleric-wis-cha'
from ten_hour_balance import load,calendar,GAPS,SESSIONS,CLASSES,OUT
parameters=json.loads((OUT/'ten_hour_selected_parameters.json').read_text())
trajectories=load('ten_hour_independent_dense.csv',list(range(6,14)))
print('Independent games:',trajectories.groupby(['class','seed_index']).ngroups,'Rows:',len(trajectories))


Independent games: 48 Rows: 4800


## Results

### 2. Recompute the equal-level growth comparison


In [2]:
comparison=[]
for name,threshold,power,amplitude in [('previous_12h',6000,1,4),('linear_10h',6000,1,2),('recommended_10h',6000,.8,2)]:
    result=calendar(trajectories,threshold,power,amplitude)
    for gap in [8.,12.]:
        index=list(itertools.product(GAPS,SESSIONS)).index((gap,12.))
        comparison.append(dict(variant=name,gap_h=gap,growth_gap_pct=result['progress_gap'][-1,index]))
comparison=pd.DataFrame(comparison)
assert np.isclose(comparison[(comparison.variant=='recommended_10h')&(comparison.gap_h==12)].growth_gap_pct.iloc[0],5.506151326265263)
print(comparison.round(4).to_string(index=False))


        variant  gap_h  growth_gap_pct
   previous_12h    8.0          2.9726
   previous_12h   12.0         10.6188
     linear_10h    8.0          2.9726
     linear_10h   12.0          5.9296
recommended_10h    8.0          2.9726
recommended_10h   12.0          5.5062


### 3. Inspect the effects and cap exceptions


In [3]:
review=json.loads((OUT/'ten_hour_validation.json').read_text())
print(pd.DataFrame.from_dict(review['level100_effects'],orient='index').round(4).to_string())
print(json.dumps(review['cap_checks'],indent=2))
assert review['app_source_hash_unchanged']
assert review['calendar_independent_implementation_match']


           cap_h  raw_proc_pct  search_s  sale_bonus_pct
CLERIC    9.1359       28.1765    4.6858         19.8333
MAGE      9.1353       30.0000    4.6858          6.3000
PALADIN   9.0824       24.6558    4.6892         19.9000
RANGER    9.1357       28.2490    4.0000          6.2667
ROGUE     9.0993       24.3921    4.0008          6.4000
WARRIOR  10.0000       24.6492    4.6892          6.5333
{
  "cap_h": {
    "specialist_at_cap": 8,
    "specialist_count": 8,
    "other_at_cap": 0,
    "other_count": 40
  },
  "raw_proc_pct": {
    "specialist_at_cap": 8,
    "specialist_count": 8,
    "other_at_cap": 0,
    "other_count": 40
  },
  "search_s": {
    "specialist_at_cap": 15,
    "specialist_count": 16,
    "other_at_cap": 0,
    "other_count": 32
  },
  "sale_bonus_pct": {
    "specialist_at_cap": 13,
    "specialist_count": 16,
    "other_at_cap": 0,
    "other_count": 32
  }
}


## Takeaways

전사는 장기 방치, DEX 직업은 잦은 사냥, 마법사는 스킬 빈도, CHA 직업은 판매 가치로 구분한다. 성직자는 WIS/CHA로 바뀌어 스킬·판매 혼합형이 되었다. 성직자 12개 경로를 다시 재생했고 다른 직업의 검증된 경로는 재사용했다. HP 후보 탐색 최저값은 기준 6,250/지수 0.8이지만, 0.1%p 미만 차이에 기준을 추가로 바꾸지 않고 6,000을 권장한다. 보정 기준의 비특화 추가시간 75분을 독립 표본 1개가 1.23분 초과했으며, DEX/CHA 저성장 경로도 상한 직전인 예외가 있었다. 전체 난수 공간이나 실제 이용자 집단의 전역 최적값은 아니다.
